# Galilean IMU preintegration with a left-invariant error

This guide develops IMU preintegration on the Galilean group $\mathrm{Gal}(3)$, including the bias-coupled geometry of $\mathrm{Gal}(3)\ltimes\mathfrak{gal}(3)$. The formulation follows GTSAM's manifold convention: tangent increments update a group element on the right, and local coordinates define a left-invariant error.

The construction is adapted from Delama, Fornasier, Mahony, and Weiss, [*Equivariant IMU Preintegration with Biases: a Galilean Group Approach*](https://arxiv.org/abs/2411.05548). The paper uses a right-invariant navigation error and applies its bias correction on the left. Here we rederive the propagation for the opposite convention; the paper's covariance and bias-update matrices therefore cannot be copied unchanged.

For background, see the [`Gal3` guide](../../geometry/doc/Gal3.ipynb), the [`Gal3ImuEKF` guide](Gal3ImuEKF.ipynb), the [`NavState` guide](NavState.ipynb), and the standard [`ImuFactor` guide](ImuFactor.ipynb).

GTSAM Copyright 2010-2022, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved

Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/navigation/doc/GalileanImuFactor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install GTSAM from pip if running in Google Colab
try:
    import google.colab
    %pip install --quiet gtsam-develop
except ImportError:
    pass  # Not in Colab

In [ ]:
import numpy as np
import gtsam

## 1. Conventions: left-invariant error means an update on the right

We fix the convention before introducing the dynamics. For a matrix Lie group $G$ with Lie algebra $\mathfrak g$, GTSAM uses

| Operation | Definition | Consequence |
|---|---|---|
| Retraction | $X\oplus\delta = X\operatorname{Exp}(\delta)$ | A tangent increment is applied on the **right**. |
| Local coordinates | $\operatorname{Local}(X,Y)=\operatorname{Log}(X^{-1}Y)$ | The displacement is expressed in the local frame of $X$. |
| Error about $\hat X$ | $X=\hat X\operatorname{Exp}(\epsilon)$ | $\epsilon=\operatorname{Local}(\hat X,X)$. |

The group error $\hat X^{-1}X$ is **left-invariant**: replacing both arguments by $AX$ and $A\hat X$ leaves it unchanged. Thus, throughout this guide, *left-invariant error* and *right perturbation* describe the same convention.

The paper instead uses the navigation error $\Upsilon\hat\Upsilon^{-1}$, which is right-invariant, and its first-order correction multiplies $\hat\Upsilon$ on the left. We will retain its physical model and bias-inclusive symmetry, but rederive all convention-dependent equations using $\hat\Upsilon^{-1}\Upsilon$.

> **Terminology warning.** The phrase *left-trivialized tangent group* describes the semidirect-product group used to couple navigation and bias. It does not select a left- or right-invariant estimation error.

## 2. The Galilean group

A Galilean element stores rotation, velocity, position, and elapsed time:

$$
X=(R,v,p,t)
\quad\longleftrightarrow\quad
\mathbf X=
\begin{bmatrix}
R & v & p\\
0 & 1 & t\\
0 & 0 & 1
\end{bmatrix}.
$$

Matrix multiplication gives

$$
(R_1,v_1,p_1,t_1)(R_2,v_2,p_2,t_2)
=\left(R_1R_2,\;v_1+R_1v_2,\;p_1+R_1p_2+t_2v_1,\;t_1+t_2\right).
$$

The term $t_2v_1$ is the defining Galilean coupling: during the second interval, the velocity accumulated by the first interval advances position. The inverse is

$$
(R,v,p,t)^{-1}=\left(R^\top,-R^\top v,-R^\top(p-tv),-t\right).
$$

Using GTSAM's tangent ordering, an algebra element is

$$
x=(\omega,\nu,\rho,\alpha)\in\mathbb R^{10},
\qquad
x^\wedge=
\begin{bmatrix}
\omega^\wedge & \nu & \rho\\
0 & 0 & \alpha\\
0 & 0 & 0
\end{bmatrix}.
$$

For an integrated IMU increment, $\omega$ is an angle, $\nu$ a velocity increment, $\rho$ a position increment, and $\alpha$ an elapsed time. Before multiplication by $\Delta t$, the corresponding rate vector has units $({\rm rad/s}, {\rm m/s^2}, {\rm m/s}, 1)$.

### Exponential, Jacobians, and adjoint

Let $J_L(\omega)$ be the $SO(3)$ left Jacobian and $\Gamma_2(\omega)$ its second integral. Then

$$
\operatorname{Exp}(x)=
\left(
\operatorname{Exp}(\omega),
J_L(\omega)\nu,
J_L(\omega)\rho+\alpha\Gamma_2(\omega)\nu,
\alpha
\right).
$$

The $\alpha\Gamma_2(\omega)\nu$ term produces the familiar $\tfrac12a\Delta t^2$ when rotation is zero. For the complete 10D algebra, define the right Jacobian $J_R(x)$ by

$$
\operatorname{Exp}(x+\delta x)
=\operatorname{Exp}(x)
 \operatorname{Exp}(J_R(x)\delta x)+O(\|\delta x\|^2).
$$

This is the Jacobian required when both the state perturbation and the increment are applied on the right. It is related to the complete Galilean left Jacobian by

$$J_R(x)=\operatorname{Ad}_{\operatorname{Exp}(-x)}J_L(x).$$

For $X=(R,v,p,t)$, the adjoint in the ordering $(\omega,\nu,\rho,\alpha)$ is

$$
\operatorname{Ad}_X=
\begin{bmatrix}
R & 0 & 0 & 0\\
v^\wedge R & R & 0 & 0\\
(p-tv)^\wedge R & -tR & R & v\\
0 & 0 & 0 & 1
\end{bmatrix}.
$$

We write $\operatorname{ad}_x y=[x,y]$ for the algebra adjoint and use the identities $\operatorname{Ad}_{X^{-1}}=\operatorname{Ad}_X^{-1}$ and $\operatorname{ad}_x y=-\operatorname{ad}_y x$.

## 3. Coordinate maps used by GTSAM

Three coordinate spaces occur in the derivation:

1. $\mathfrak{gal}(3)$ uses $e=(\delta\theta,\delta v,\delta p,\delta t)\in\mathbb R^{10}$.
2. The bias-coupled error uses $(e,d)\in\mathbb R^{20}$.
3. `NavState` local coordinates use $n=(\delta\theta,\delta p,\delta v)\in\mathbb R^9$.

The selection and permutation from a Galilean tangent error to a `NavState` tangent error is

$$
n=P_Ne,\qquad
P_N=
\begin{bmatrix}
I_3&0&0&0\\
0&0&I_3&0\\
0&I_3&0&0
\end{bmatrix}
\in\mathbb R^{9\times10}.
$$

The final column removes elapsed time, which is known exactly from the measurement timestamps. Notice that this projection applies to **local errors and covariances**. The mean Galilean element is converted to `NavState` as $(R,p,v)$; its position is not obtained by blindly selecting entries from $\operatorname{Log}(\Upsilon)$.

There is a second ordering conversion for physical biases. `gtsam::imuBias::ConstantBias::vector()` stores

$$\delta\beta=(\delta b_a,\delta b_g)\in\mathbb R^6,$$

whereas a Galilean input stores angular velocity before acceleration. Define

$$
S_\beta=
\begin{bmatrix}
0&I_3\\
I_3&0\\
0&0\\
0&0
\end{bmatrix}
\in\mathbb R^{10\times6},\qquad
\delta b=S_\beta\delta\beta.
$$

The last four rows correspond to the virtual velocity and time biases. They are necessary to define the symmetry but are zero for a physical six-axis IMU.

## 4. Mean preintegration

Embed one IMU sample and its bias in $\mathbb R^{10}$ as

$$
\tilde w_k=(\tilde\omega_k,\tilde a_k,0,1),\qquad
\hat b_k=(\hat b_{g,k},\hat b_{a,k},0,0).
$$

For $h=\Delta t_k$, define

$$
q_k=\tilde w_k-\hat b_k,\qquad
x_k=q_kh,\qquad
V_k=\operatorname{Exp}(x_k).
$$

Starting with $\hat\Upsilon_{ii}=I$, the preintegrated mean is updated on the right:

$$
\boxed{\hat\Upsilon_{i,k+1}=\hat\Upsilon_{ik}V_k}.
$$

The group law automatically performs all four component updates. In particular, old velocity advances position by $h\,\Delta v_{ik}$, while the exponential contributes the rotation-aware acceleration integral. No separate Euler position update is needed.

For navigation states $T_i,T_j\in\mathrm{Gal}(3)$ at keyframes, gravity and the passage of time are collected in a known left increment $\Gamma_{ij}$, giving

$$
T_j=\Gamma_{ij}T_i\Upsilon_{ij},
\qquad
\Upsilon_{ij}=T_i^{-1}\Gamma_{ij}^{-1}T_j.
$$

This separation is important: gravity acts in the navigation frame on the left, while bias-corrected IMU increments act in the body/local frame on the right.

## 5. Coupling navigation and bias

The left-trivialized tangent group is the semidirect product

$$
\mathcal G=\mathrm{Gal}(3)\ltimes\mathfrak{gal}(3),
$$

with multiplication and inverse

$$
(A,a)(B,b)=(AB,a+\operatorname{Ad}_A b),\qquad
(A,a)^{-1}=(A^{-1},-\operatorname{Ad}_{A^{-1}}a).
$$

A physical preintegration state $(\Upsilon,b)$ is represented in this group by

$$X(\Upsilon,b)=(\Upsilon,-\operatorname{Ad}_{\Upsilon}b).$$

Let $X$ be the true state and $\hat X$ the nominal state. The GTSAM-compatible augmented error is

$$
Z=\hat X^{-1}X=(E,\beta),\qquad
E=\hat\Upsilon^{-1}\Upsilon,\qquad
\beta=\hat b-\operatorname{Ad}_E b.
$$

Normal coordinates on the tangent group are

$$
\epsilon=(e,d)=\operatorname{Log}_{\mathcal G}(Z),\qquad
e=\operatorname{Log}(E),\qquad
d=J_L(e)^{-1}\beta.
$$

Equivalently, $X=\hat X\operatorname{Exp}_{\mathcal G}(e,d)$. Linearizing the relation between these coordinates and the ordinary bias difference gives

$$
b-\hat b=-d-\operatorname{ad}_e\hat b+O(\|\epsilon\|^2).
$$

This term is the geometric coupling: the second ten coordinates are not simply $b-\hat b$. A navigation perturbation changes the frame in which the bias error is represented.

## 6. Left-invariant discrete error dynamics

We now derive the matrices used for uncertainty propagation. Adopt the measurement convention

$$
\tilde w_k=w_k+\eta_{w,k},\qquad
0=\tilde\tau_k=\tau_k+\eta_{\tau,k}.
$$

Thus the true bias evolves as $b_{k+1}=b_k-\eta_{\tau,k}h$, while the nominal bias is constant over one preintegration. Both $\eta_w$ and $\eta_\tau$ are 10D vectors whose virtual components have zero covariance for a physical IMU.

For one nominal increment define

$$
V=\operatorname{Exp}(x),\qquad
C=J_R(x)h,\qquad
F=\operatorname{Ad}_{V^{-1}}-C\operatorname{ad}_{\hat b},
\qquad x=(\tilde w-\hat b)h.
$$

To first order, the corrected true input differs from the nominal input by

$$
\delta q=d+\operatorname{ad}_e\hat b-\eta_w
=d-\operatorname{ad}_{\hat b}e-\eta_w.
$$

Substituting $\Upsilon=\hat\Upsilon\operatorname{Exp}(e)$ into the right mean update gives

$$
e^+=Fe+Cd-C\eta_w.
$$

Using $d^+\simeq\hat b-\operatorname{Ad}_{\operatorname{Exp}(e^+)}b^+$ then gives

$$
d^+=\operatorname{ad}_{\hat b}(F-I)e
+(I+\operatorname{ad}_{\hat b}C)d
-\operatorname{ad}_{\hat b}C\eta_w+h\eta_\tau.
$$

Therefore, with $\epsilon=(e,d)$ and $\eta=(\eta_w,\eta_\tau)$,

$$
\boxed{\epsilon^+=A_{\mathrm{LI}}\epsilon+B_{\mathrm{LI}}\eta},
$$

where

$$
A_{\mathrm{LI}}=
\begin{bmatrix}
F&C\\
\operatorname{ad}_{\hat b}(F-I)&I+\operatorname{ad}_{\hat b}C
\end{bmatrix},\qquad
B_{\mathrm{LI}}=
\begin{bmatrix}
-C&0\\
-\operatorname{ad}_{\hat b}C&hI
\end{bmatrix}.
$$

Every displayed block is $10\times10$. In particular, the upper-left block is generally **not** the identity: $\operatorname{Ad}_{V^{-1}}$ transports a right perturbation through the next right-applied increment. This is the most visible difference from the paper's right-invariant propagation.

### Covariance propagation

Let $\Sigma_k\in\mathbb R^{20\times20}$ be the covariance of $(e,d)$. If $Q_c$ contains continuous-time noise densities, the sampled rate-noise covariance is

$$
Q_d=\frac{1}{h}
\operatorname{diag}(Q_w,Q_\tau),
$$

and the first-order discrete propagation is

$$
\boxed{
\Sigma_{k+1}=A_{\mathrm{LI}}\Sigma_kA_{\mathrm{LI}}^\top
+B_{\mathrm{LI}}Q_dB_{\mathrm{LI}}^\top}.
$$

For a six-axis IMU, $Q_w$ has nonzero gyroscope and accelerometer blocks only, and $Q_\tau$ has nonzero gyroscope- and accelerometer-bias random-walk blocks only. All virtual-input and virtual-bias rows and columns are zero. If a caller already supplies per-sample rather than continuous-time covariances, the $1/h$ conversion must not be applied a second time.

The three-way IMU factor does not itself constrain temporal bias evolution. It uses the navigation uncertainty implied by this model, while a separate bias between-factor supplies the chosen random-walk model. A future combined-bias factor could instead consume the full physical $15\times15$ projection.

For the concrete three-way factor, the bias is conditioned on its linearization value during preintegration: set $Q_\tau=0$ and initialize the bias error to zero. Restricting the augmented recursion to this conditioned subspace cancels the bias-coupling terms in the navigation block, leaving

$$
\Sigma_{e,k+1}=\operatorname{Ad}_{V_k^{-1}}\Sigma_{e,k}\operatorname{Ad}_{V_k^{-1}}^\top+C_k\frac{Q_w}{h}C_k^\top.
$$

Consequently an implementation of the three-way factor can propagate this conditional $10\times10$ covariance directly, without storing the full $20\times20$ matrix. The separate bias evolution factor carries $Q_\tau$; a combined-bias factor would instead retain the full augmented covariance and its state-bias cross terms.

## 7. Bias correction must also act on the right

Preintegration is performed once at a linearization bias $\hat b$. When optimization proposes $\hat b+\delta b$, recomputing every IMU sample would be wasteful. Define $J_k\in\mathbb R^{10\times10}$ by

$$
\Upsilon_k(\hat b+\delta b)
\simeq\hat\Upsilon_k(\hat b)
\operatorname{Exp}(J_k\delta b).
$$

This correction is on the right, matching the retraction convention. For $V_k=\operatorname{Exp}(x_k)$ and $C_k=J_R(x_k)h$, transport of the old correction through the new increment gives

$$
\boxed{J_{k+1}=\operatorname{Ad}_{V_k^{-1}}J_k-C_k},
\qquad J_i=0.
$$

The minus sign follows from differentiating $\tilde w-b$. For GTSAM's physical bias coordinates, use

$$
J_k^{\mathrm{phys}}=J_kS_\beta\in\mathbb R^{10\times6},
$$

and apply

$$
\Upsilon_k(\beta)\simeq
\hat\Upsilon_k
\operatorname{Exp}
\left(J_k^{\mathrm{phys}}(\beta-\hat\beta)\right).
$$

If the full tangent-group sensitivity is desired, the associated first-order bias coordinate is $d=(-I+\operatorname{ad}_{\hat b}J_k)\delta b$. The navigation factor needs only the upper, Galilean correction $J_k$.

## 8. From the Galilean uncertainty to a GTSAM factor

Let $\Sigma_{ee}$ be the upper-left $10\times10$ block of the augmented covariance. The factor's `NavState` residual covariance is

$$
\Sigma_N=P_N\Sigma_{ee}P_N^\top\in\mathbb R^{9\times9}.
$$

Likewise, the first-order bias sensitivity of the local navigation error is

$$
J_N=P_NJ_kS_\beta\in\mathbb R^{9\times6}.
$$

For the mean, first apply the Galilean bias correction and then construct

$$
\Delta X_{ij}=
\operatorname{NavState}
(\Delta R_{ij},\Delta p_{ij},\Delta v_{ij}).
$$

The known gravity and elapsed-time terms combine this delta with $X_i$ to form `predictedState_j`. The residual used by the existing GTSAM preintegration machinery is

$$
r_{ij}=\operatorname{Local}_{\mathrm{NavState}}
(X_j,\operatorname{predictedState}_j)
=X_j.\texttt{localCoordinates}(\operatorname{predictedState}_j).
$$

In the C++ interface this is `NavState::localCoordinates`; the argument order above is intentional because GTSAM's IMU residual asks how the measured state retracts to the prediction.

Conceptually this is a three-variable relation $(X_i,X_j,\beta_i)$. The current `GalileanImuFactor` alias uses `ImuFactorT`, so each `NavState` is exposed as separate pose and velocity keys and the concrete graph factor has five keys. In both views there is one bias variable for the entire interval. Bias evolution remains a separate factor, exactly as for the standard `ImuFactor`.

## 9. Intended use

A user-facing workflow mirrors the existing IMU factors:

1. Construct preintegration parameters and a `PreintegratedImuMeasurementsG` at the current bias estimate.
2. Integrate each accelerometer/gyroscope sample. Each call advances the Galilean mean on the right and propagates the left-invariant uncertainty.
3. Construct a `GalileanImuFactor` between the two pose/velocity pairs and the interval's bias key. During optimization, use the right-applied first-order bias correction rather than reintegrating immediately.
4. Add a separate factor between consecutive bias keys when bias random-walk evolution is part of the model.

In schematic C++ form:

```cpp
PreintegratedImuMeasurementsG pim(params, biasHat);
for (const ImuSample& sample : samples) {
  pim.integrateMeasurement(sample.acceleration, sample.angularRate,
                           sample.deltaT);
}
graph.emplace_shared<GalileanImuFactor>(
    X(i), V(i), X(j), V(j), B(i), pim);
```

The API is deliberately familiar; the essential differences are internal geometric choices.

## 10. Consistency checks for the formulation

The following statements should all remain true together:

- The mean update is $\Upsilon^+=\Upsilon\operatorname{Exp}(x)$, never $\operatorname{Exp}(x)\Upsilon$.
- The navigation error is $\hat\Upsilon^{-1}\Upsilon$, never $\Upsilon\hat\Upsilon^{-1}$.
- Transport through an increment uses $\operatorname{Ad}_{V^{-1}}$. Consequently the left-invariant transition is not generally the identity.
- Input perturbations use the complete Galilean right Jacobian $J_R(x)$.
- Bias correction is $\hat\Upsilon\operatorname{Exp}(J\delta b)$, not $\operatorname{Exp}(J\delta b)\hat\Upsilon$.
- GTSAM bias coordinates are $(b_a,b_g)$, while Galilean input coordinates begin with $(b_g,b_a)$.
- The 10D-to-9D covariance conversion removes time and swaps the velocity and position blocks.
- The preintegration interval is accumulated alongside the Galilean mean; time is not an uncertain optimized coordinate.

Changing any one of these conventions requires rederiving the associated Jacobians and covariance transport.

## References

- G. Delama, A. Fornasier, R. Mahony, and S. Weiss, [*Equivariant IMU Preintegration with Biases: a Galilean Group Approach*](https://arxiv.org/abs/2411.05548), IEEE Robotics and Automation Letters, 2025.
- [`gtsam::Gal3`](../../geometry/doc/Gal3.ipynb): group operations, exponential and logarithmic maps, and adjoints.
- [`Gal3ImuEKF`](Gal3ImuEKF.ipynb): Galilean state propagation in an invariant EKF.
- [`NavState`](NavState.ipynb): GTSAM's navigation manifold and local-coordinate convention.
- [`ImuFactor`](ImuFactor.ipynb): standard preintegration and factor-graph usage.